# Metabolite meta-analysis
Here, we are interested in comparing our findings to other papers that have analyzed the metabolite hits we've gotten. 
| Author | Lab | Cohort | Title | Assigned number |
|-------------|------------------|------------|------|-------|
|Forslund 2021    | Bork P. | MetaCardis | Combinatorial, additive and dose-dependent drug-microbiome associations | 1 |
|Fromentin 2022    | Pedersen O.| MetaCardis | Microbiome and metabolome features of the cardiometabolic disease spectrum | 1 |
| Wu 2025    | Bäckhed F. | Swedish IGT and SCAPIS | Microbiome-metabolome dynamics associated with impaired glucose control and responses to lifestyle changes | 2 |
| Li 2026   | Qi Q. | 10 cohorts, including SOL | Circulating metabolites, genetics and lifestyle factors in relation to future risk of  type 2 diabetes | 3 |
| Yousri 2021   | Yousri N. | Qatari + QBB | Metabolic and Metabo-Clinical Signatures of Type 2 Diabetes, Obesity, Retinopathy, and Dyslipidemia | 4 |

In [2]:
import pandas as pd
import numpy as np
import tabula
import matplotlib.pyplot as plt
import seaborn as sns
import arivale_data_interface as adi

# Read in Arivale results

In [3]:
metab_betas = pd.read_csv("../data/rh_metabs_betas.csv")
metab_qs = pd.read_csv("../data/rh_metabs_qs.csv")

In [3]:
# metab_betas

In [5]:
# Are in the same order
#metab_betas.CHEMICAL_NAME == metab_qs.CHEMICAL_NAME

# metab_betas.set_index('CHEMICAL_NAME', inplace=True)
# metab_qs.set_index('CHEMICAL_NAME', inplace=True)
# # Create and apply mask
# metab_qs_mask = metab_qs > 0.1
# metab_betas[metab_qs_mask] = np.nan

In [5]:
# metab_betas.reset_index(inplace=True)

In [6]:
anns_snapshot = pd.read_csv("../data/metabolites_snapshot_updated.csv")

Going to add metabolon columns, so we can annotate our heatmap.

In [7]:
anns = anns_snapshot[['CHEMICAL_NAME','SUB_PATHWAY','SUPER_PATHWAY']].copy()
anns.set_index('CHEMICAL_NAME', inplace=True)

In [8]:
metab_betas = metab_betas.merge(anns, how='left', on='CHEMICAL_NAME')

Merge the results from literature onto `metab_betas`

In [9]:
plot_labs = metab_betas.CHEMICAL_NAME.tolist() # Create a smaller name for plots
metab_betas['plot_name'] = plot_labs

In [10]:
# Replace special characters
metab_betas.CHEMICAL_NAME = metab_betas.CHEMICAL_NAME.str.replace('-', '_')
metab_betas.CHEMICAL_NAME = metab_betas.CHEMICAL_NAME.str.replace(' ', '_')
metab_betas.CHEMICAL_NAME = metab_betas.CHEMICAL_NAME.str.replace('(', '')
metab_betas.CHEMICAL_NAME = metab_betas.CHEMICAL_NAME.str.replace(')', '')

In [11]:
# This must come after the group above; for X - ##### metabs
metab_betas.CHEMICAL_NAME = metab_betas.CHEMICAL_NAME.str.replace('___', '_')

In [12]:
sig_metabs_name_clean = metab_betas.CHEMICAL_NAME.tolist()

In [13]:
metab_betas

,CHEMICAL_NAME,abx_three,abx_twelve,chol_meds,blood_sugar_meds,SUB_PATHWAY,SUPER_PATHWAY,plot_name
0,3_4_hydroxyphenyllactate,-0.213787,-0.117430,-0.070555,0.093941,Tyrosine Metabolism,Amino Acid,3-(4-hydroxyphenyl)lactate
1,glutarate_C5_DC,0.302153,0.081044,-0.005269,0.246936,"Fatty Acid, Dicarboxylate",Lipid,glutarate (C5-DC)
2,alpha_ketobutyrate,-0.235046,-0.018865,0.068983,0.489881,"Methionine, Cysteine, SAM and Taurine Metabolism",Amino Acid,alpha-ketobutyrate
3,3_phenylpropionate_hydrocinnamate,-0.343529,-0.179521,-0.082576,-0.061306,Benzoate Metabolism,Xenobiotics,3-phenylpropionate (hydrocinnamate)
4,hippurate,-0.262174,-0.196912,-0.173505,-0.159983,Benzoate Metabolism,Xenobiotics,hippurate
5,4_methyl_2_oxopentanoate,-0.208599,-0.106307,0.074137,0.544634,"Leucine, Isoleucine and Valine Metabolism",Amino Acid,4-methyl-2-oxopentanoate
6,alpha_hydroxyisocaproate,-0.213506,-0.101245,-0.022021,0.218990,"Leucine, Isoleucine and Valine Metabolism",Amino Acid,alpha-hydroxyisocaproate
7,3_hydroxyoctanoate,-0.228937,-0.147866,0.070133,0.887619,"Fatty Acid, Monohydroxy",Lipid,3-hydroxyoctanoate
8,phenyllactate_PLA,-0.240545,-0.134656,-0.039017,0.094937,Phenylalanine Metabolism,Amino Acid,phenyllactate (PLA)
9,acetylcarnitine_C2,-0.273789,-0.061022,-0.013758,0.058746,"Fatty Acid Metabolism (Acyl Carnitine, Short C...",Lipid,acetylcarnitine (C2)


## MetaCardis PatientGroup coefficients

In [14]:
metacardis_metS = pd.read_csv("../data/metacardis_metS.csv")
metacardis_ob = pd.read_csv("../data/metacardis_severe_ob.csv")
metacardis_t2dm = pd.read_csv("../data/metacardis_t2dm.csv")

In [15]:
# metacardis_ob

In [15]:
def clean_mc_dfs(df):
    # Keep only sig hits
    #df.loc[df['q'] >= 0.1, 'beta'] = np.nan
    df.rename(columns={'feature':'CHEMICAL_NAME'}, inplace=True)
    df.CHEMICAL_NAME = df.CHEMICAL_NAME.str.replace('metabolite_', '')
    df = df[['CHEMICAL_NAME', 'beta']].copy()
    return df

In [16]:
metacardis_metS_clean = clean_mc_dfs(metacardis_metS)
metacardis_ob_clean = clean_mc_dfs(metacardis_ob)
metacardis_t2dm_clean = clean_mc_dfs(metacardis_t2dm)

In [17]:
metacardis_ob_clean

,CHEMICAL_NAME,beta
0,2_hydroxy_3_methylvalerate,-0.441312
1,2_hydroxybutyrate_2_hydroxyisobutyrate,0.270651
2,2_piperidinone,-0.101765
3,3_3_hydroxyphenylpropionate,0.089162
4,3_4_hydroxyphenyllactate_HPLA,-0.158681
5,3_hydroxy_2_ethylpropionate,-0.082072
6,3_hydroxydecanoate,-0.232863
7,3_hydroxylaurate,-0.153007
8,3_hydroxymyristate,-0.186809
9,3_hydroxyoctanoate,-0.276438


In [18]:
metab_betas = metab_betas.merge(
    metacardis_metS_clean, 
    on='CHEMICAL_NAME', 
    how='left'
).rename(columns={'beta': 'metacardis_metabolic_syndrome'})

metab_betas = metab_betas.merge(
    metacardis_ob_clean, 
    on='CHEMICAL_NAME', 
    how='left'
).rename(columns={'beta': 'metacardis_severe_ob'})

metab_betas = metab_betas.merge(
    metacardis_t2dm_clean, 
    on='CHEMICAL_NAME', 
    how='left'
).rename(columns={'beta': 'metacardis_t2dm'})

In [19]:
metab_betas

,CHEMICAL_NAME,abx_three,abx_twelve,chol_meds,blood_sugar_meds,SUB_PATHWAY,SUPER_PATHWAY,plot_name,metacardis_metabolic_syndrome,metacardis_severe_ob,metacardis_t2dm
0,3_4_hydroxyphenyllactate,-0.213787,NaN,NaN,NaN,Tyrosine Metabolism,Amino Acid,3-(4-hydroxyphenyl)lactate,NaN,NaN,NaN
1,glutarate_C5_DC,0.302153,NaN,NaN,NaN,"Fatty Acid, Dicarboxylate",Lipid,glutarate (C5-DC),NaN,NaN,NaN
2,alpha_ketobutyrate,-0.235046,NaN,NaN,0.489881,"Methionine, Cysteine, SAM and Taurine Metabolism",Amino Acid,alpha-ketobutyrate,0.297633,0.629332,0.658268
3,3_phenylpropionate_hydrocinnamate,-0.343529,NaN,NaN,NaN,Benzoate Metabolism,Xenobiotics,3-phenylpropionate (hydrocinnamate),-0.360772,-0.545492,-0.510206
4,hippurate,-0.262174,NaN,NaN,NaN,Benzoate Metabolism,Xenobiotics,hippurate,-0.285934,NaN,-0.458875
5,4_methyl_2_oxopentanoate,-0.208599,NaN,NaN,0.544634,"Leucine, Isoleucine and Valine Metabolism",Amino Acid,4-methyl-2-oxopentanoate,0.295596,NaN,NaN
6,alpha_hydroxyisocaproate,-0.213506,NaN,NaN,NaN,"Leucine, Isoleucine and Valine Metabolism",Amino Acid,alpha-hydroxyisocaproate,NaN,NaN,NaN
7,3_hydroxyoctanoate,-0.228937,NaN,NaN,0.887619,"Fatty Acid, Monohydroxy",Lipid,3-hydroxyoctanoate,NaN,NaN,NaN
8,phenyllactate_PLA,-0.240545,NaN,NaN,NaN,Phenylalanine Metabolism,Amino Acid,phenyllactate (PLA),-0.278314,NaN,NaN
9,acetylcarnitine_C2,-0.273789,NaN,NaN,NaN,"Fatty Acid Metabolism (Acyl Carnitine, Short C...",Lipid,acetylcarnitine (C2),NaN,NaN,NaN


## Wu 2025: metabolites and pre(diabetes)

Metabolite metadata (get names; their format is M#####)

In [20]:
wu_metab_meta = pd.read_excel("../data/Wu2025_Supplement.xlsx", sheet_name=2, header=0)
wu_metab_dict = {key: value for key, value in zip(wu_metab_meta.Metabolites, wu_metab_meta.BIOCHEMICAL)}

In [21]:
wu_metab_or = pd.read_excel("../data/Wu2025_Supplement.xlsx", sheet_name=8, header=1)

# In this study, they compute ORs along the glucose intolerance spectrum
wu_metab_or.rename(columns={'Odds ratios':'IFG_OR',         # Impaired fasting
                            'Odds ratios.1':'IGT_OR',       # Impaired glucose tolerance
                            'Odds ratios.2':'CGI_T2D_OR',   # Combined glucose interolance and T2D
                            'Metabolites':'CHEMICAL_NAME'}, inplace=True)
wu_metab_or['CHEMICAL_NAME'] = wu_metab_or['CHEMICAL_NAME'].apply(lambda x: wu_metab_dict[x])

# Noticed that HPLA is in parenthesis; change this
wu_metab_or.CHEMICAL_NAME = wu_metab_or.CHEMICAL_NAME.str.replace(' (HPLA)', '')
# wu_metab_or.CHEMICAL_NAME = wu_metab_or.CHEMICAL_NAME.str.replace('N-lactoyl leucine', '1-carboxyethylleucine')

wu_metab_or.rename(columns={'CHEMICAL_NAME':'plot_name'}, inplace=True)

### Select overlapping metabolites with IFG group

In [22]:
wu_metab_ifg = wu_metab_or[['plot_name','IFG_OR','Adjusted P Values']]
wu_metab_ifg = wu_metab_ifg[wu_metab_ifg['Adjusted P Values'] < 0.1].drop(columns=['Adjusted P Values'])
wu_metab_ifg = wu_metab_ifg[wu_metab_ifg['plot_name'].isin(plot_labs)]
wu_metab_ifg

,plot_name,IFG_OR
14,2-hydroxybutyrate/2-hydroxyisobutyrate,1.3376
20,alpha-ketobutyrate,1.3151
422,indolepropionate,0.6973
477,3-phenylpropionate (hydrocinnamate),0.7129


### IGT group

In [23]:
wu_metab_igt = wu_metab_or[['plot_name','IGT_OR','Adjusted P Values.1']]
wu_metab_igt = wu_metab_igt[wu_metab_igt['Adjusted P Values.1'] < 0.1].drop(columns=['Adjusted P Values.1'])
wu_metab_igt = wu_metab_igt[wu_metab_igt['plot_name'].isin(plot_labs)]
wu_metab_igt

,plot_name,IGT_OR
14,2-hydroxybutyrate/2-hydroxyisobutyrate,1.7114
20,alpha-ketobutyrate,1.6246
21,4-methyl-2-oxopentanoate,1.7317
108,alpha-hydroxyisocaproate,1.2672
161,isoursodeoxycholate,1.1877
172,3-hydroxymyristate,1.4277
174,3-hydroxylaurate,1.4029
193,alpha-hydroxyisovalerate,1.1735
194,acetylcarnitine (C2),1.2609
226,3-hydroxydecanoate,1.3008


### CGI / T2D

In [24]:
wu_metab_gci = wu_metab_or[['plot_name','CGI_T2D_OR','Adjusted P Values.2']]
wu_metab_gci = wu_metab_gci[wu_metab_gci['Adjusted P Values.2'] < 0.1].drop(columns=['Adjusted P Values.2'])
wu_metab_gci = wu_metab_gci[wu_metab_gci['plot_name'].isin(plot_labs)]
wu_metab_gci

,plot_name,CGI_T2D_OR
14,2-hydroxybutyrate/2-hydroxyisobutyrate,2.1925
20,alpha-ketobutyrate,2.1387
21,4-methyl-2-oxopentanoate,2.1023
108,alpha-hydroxyisocaproate,1.5315
125,2-hydroxy-3-methylvalerate,1.4679
161,isoursodeoxycholate,1.4018
172,3-hydroxymyristate,1.3842
174,3-hydroxylaurate,1.3759
193,alpha-hydroxyisovalerate,1.3352
194,acetylcarnitine (C2),1.3343


In [25]:
# Merge onto pivot!
metab_betas = metab_betas.merge(wu_metab_gci, on='plot_name', how='left')
metab_betas = metab_betas.merge(wu_metab_igt, on='plot_name', how='left')
metab_betas = metab_betas.merge(wu_metab_ifg, on='plot_name', how='left')

In [26]:
# metab_betas

## EPIC-Norfolk Cox Proportional Hazards

| Table | Title | Adjustments |
|-------|-------|-------------|
|S5 | Results from multivariable adjusted Cox models using metabolite levels as main exposure, resticting to at least suggestive evidence for statistical significance (p < 0.001) | aObtained from Cox-proportional hazard models adjusted for age, (sex), body mass index, waist-to-hip ratio, smoking behaviour, serum uric acid concentrations, total triglycerides, HDL-cholesterol, random glucose, serum alkaline phosphatase concentrations, serum vitamin C concentrations, systolic blood pressure, and plasma fibrinogen concentrations. *Metabolite annotation was based on in-silico predictions							
|S6 | Results from Cox propotional hazard models for metabolites defined as uniquely associated with only one endpoint | aObtained from Cox-proportional hazard models adjusted for age and sex. *Metabolite annotations was based on in-silico predictions | 
|S7 | Metabolites signficantly associated (p<5.07E-5) in logistic regression analysis using  development of two or more diseases as binary outcome. | aObtained from a general linear regression model with a binomial link function adjusting for age and sex. *Metabolite annotations was based on in-silico predictions |

In [27]:
epic_metabs_S5 = pd.read_excel("../data/pietzner2021_cox_prop_metabolites.xlsx", sheet_name=4, header=2)
epic_metabs_S7 = pd.read_excel("../data/pietzner2021_cox_prop_metabolites.xlsx", sheet_name=6, header=2)

In [28]:
# epic_metabs_S5.BIOCHEMICAL[epic_metabs_S5.BIOCHEMICAL.str.contains('leucine')]

In [29]:
# S5
epic_metabs_S5.BIOCHEMICAL = epic_metabs_S5.BIOCHEMICAL.str.replace(r"[()'*]", "", regex=True)
epic_metabs_S5.BIOCHEMICAL = epic_metabs_S5.BIOCHEMICAL.str.replace(r"[:/\-\s]", "_", regex=True)

# S7
epic_metabs_S7.BIOCHEMICAL = epic_metabs_S7.BIOCHEMICAL.str.replace(r"[()'*]", "", regex=True)
epic_metabs_S7.BIOCHEMICAL = epic_metabs_S7.BIOCHEMICAL.str.replace(r"[:/\-\s]", "_", regex=True)

In [30]:
# epic_metabs_S5.BIOCHEMICAL[epic_metabs_S5.BIOCHEMICAL.str.contains('phe')]

In [31]:
epic_metabs_S5.BIOCHEMICAL = epic_metabs_S5.BIOCHEMICAL.str.replace("___", "_")
epic_metabs_S5.BIOCHEMICAL = epic_metabs_S5.BIOCHEMICAL.str.replace("hydantoin_5_propionic_acid",
                                                                        "hydantoin_5_propionate")
epic_metabs_S7.BIOCHEMICAL = epic_metabs_S7.BIOCHEMICAL.str.replace("___", "_")
epic_metabs_S7.BIOCHEMICAL = epic_metabs_S7.BIOCHEMICAL.str.replace("hydantoin_5_propionic_acid",
                                                                        "hydantoin_5_propionate")

In [32]:
epic_metabs_S5 = epic_metabs_S5.iloc[:-1]
epic_metabs_S7 = epic_metabs_S7.iloc[:-1]

epic_metabs_S5.rename(columns={'BIOCHEMICAL':'CHEMICAL_NAME'}, inplace=True)
epic_metabs_S7.rename(columns={'BIOCHEMICAL':'CHEMICAL_NAME'}, inplace=True)

In [33]:
# epic_metabs_S5['Incident endpoint'].value_counts()
endpoints = ['Type 2 diabetes']
selected_S5 = epic_metabs_S5[epic_metabs_S5['Incident endpoint'].isin(endpoints)]
S5_arivale = selected_S5[selected_S5.CHEMICAL_NAME.isin(sig_metabs_name_clean)].copy()

In [34]:
S5_arivale.loc[:,'HR'] = S5_arivale.loc[:,'Hazard ratio (95%-CI)a'].str.extract(r'^([\d.]+)')

In [35]:
S5_arivale_pivot = S5_arivale.pivot(index='CHEMICAL_NAME', columns='Incident endpoint', values='HR').reset_index()

In [36]:
S5_arivale_pivot

Incident endpoint,CHEMICAL_NAME,Type 2 diabetes
0,X_12216,0.85
1,cinnamoylglycine,0.82
2,hippurate,0.85


In [37]:
metab_betas = metab_betas.merge(
    S5_arivale_pivot, 
    on='CHEMICAL_NAME', 
    how='left').rename(columns={'Type 2 diabetes': 'EPIC_T2DM'})
metab_betas.head()

,CHEMICAL_NAME,abx_three,abx_twelve,chol_meds,blood_sugar_meds,SUB_PATHWAY,SUPER_PATHWAY,plot_name,metacardis_metabolic_syndrome,metacardis_severe_ob,metacardis_t2dm,CGI_T2D_OR,IGT_OR,IFG_OR,EPIC_T2DM
0,3_4_hydroxyphenyllactate,-0.213787,NaN,NaN,NaN,Tyrosine Metabolism,Amino Acid,3-(4-hydroxyphenyl)lactate,NaN,NaN,NaN,1.3336,NaN,NaN,NaN
1,glutarate_C5_DC,0.302153,NaN,NaN,NaN,"Fatty Acid, Dicarboxylate",Lipid,glutarate (C5-DC),NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,alpha_ketobutyrate,-0.235046,NaN,NaN,0.489881,"Methionine, Cysteine, SAM and Taurine Metabolism",Amino Acid,alpha-ketobutyrate,0.297633,0.629332,0.658268,2.1387,1.6246,1.3151,NaN
3,3_phenylpropionate_hydrocinnamate,-0.343529,NaN,NaN,NaN,Benzoate Metabolism,Xenobiotics,3-phenylpropionate (hydrocinnamate),-0.360772,-0.545492,-0.510206,0.5005,0.6767,0.7129,NaN
4,hippurate,-0.262174,NaN,NaN,NaN,Benzoate Metabolism,Xenobiotics,hippurate,-0.285934,NaN,-0.458875,0.5751,0.6626,NaN,0.85


In [38]:
S7_arivale = epic_metabs_S7[epic_metabs_S7['CHEMICAL_NAME'].isin(sig_metabs_name_clean)].copy()
S7_arivale['OR'] = S7_arivale['Odds ratio (95%-CI)a'].str.extract(r'^([\d.]+)')
S7_arivale

,CHEMICAL_NAME,SUPER_PATHWAY,SUB_PATHWAY,N,Odds ratio (95%-CI)a,p-value,OR
4,3_phenylpropionate_hydrocinnamate,Xenobiotics,Benzoate Metabolism,5519.0,0.83 (0.78; 0.89),2.250000e-08,0.83
11,isoursodeoxycholate,Lipid,Secondary Bile Acid Metabolism,5642.0,1.17 (1.10; 1.25),8.830000e-07,1.17
17,X_12216,NaN,NaN,5645.0,0.86 (0.81; 0.92),3.800000e-06,0.86
21,indolepropionate,Amino Acid,Tryptophan Metabolism,5670.0,0.87 (0.82; 0.93),1.880000e-05,0.87


In [39]:
metab_betas = metab_betas.merge(
    S7_arivale[['CHEMICAL_NAME', 'OR']], 
    on='CHEMICAL_NAME', 
    how='left').rename(columns={'OR': 'EPIC_two_dx_OR'})

In [40]:
# metab_betas

## 10 Cohorts, including SOL

In [41]:
li_t2dm = pd.read_excel("../data/Li_2026.xlsx", sheet_name = 1, skiprows=[0,1], header=[0, 1])

In [42]:
# li_t2dm.iloc[:,:16]

In [43]:
basic_bmi_waist_model = li_t2dm.loc[:, [
    ('HMDBID', 'HMDBID'),
    ('Metabolite', 'Metabolite'), 
    ('Super Pathway', 'Super Pathway'),
    ('Sub Pathway', 'Sub Pathway'),
    ('Main Model (Basic Model + BMI + waist measures)','HR (95%CI)'),
    ('Main Model (Basic Model + BMI + waist measures)','P'),
    ('Main Model (Basic Model + BMI + waist measures)','FDR'),
    ('Main Model (Basic Model + BMI + waist measures)','Het-P'),
    ('Main Model (Basic Model + BMI + waist measures)','Study-subset n')]
].copy()

basic_bmi_waist_model.columns = basic_bmi_waist_model.columns.droplevel(0)

In [44]:
# Grab list to copy
hmdb_list = (anns_snapshot[
 anns_snapshot['CHEMICAL_NAME'].isin(metab_betas['plot_name']) & 
 anns_snapshot['HMDB'].notna()])['HMDB'].tolist()

clover_list = [sub_item for item in hmdb_list for sub_item in item.split(',')]

In [45]:
found_metabs_li = basic_bmi_waist_model[basic_bmi_waist_model.HMDBID.isin(clover_list)|basic_bmi_waist_model.Metabolite.isin(plot_labs)]

In [46]:
# basic_bmi_waist_model.Metabolite[basic_bmi_waist_model.Metabolite.str.contains('piper')]

In [47]:
found_metabs_li_sig = found_metabs_li[found_metabs_li['FDR'] <0.1].copy()
found_metabs_li_sig.loc[:, 'HR'] = found_metabs_li_sig.loc[:, 'HR (95%CI)'].str.extract(r'^([\d.]+)')

In [48]:
found_metabs_li_sig_small = found_metabs_li_sig[['Metabolite', 'HR']].copy()
found_metabs_li_sig_small = found_metabs_li_sig_small.rename(
    columns={'Metabolite':'plot_name',
             'HR':'li_t2dm'})

In [49]:
metab_betas = metab_betas.merge(
    found_metabs_li_sig_small, 
    how='left',
    on='plot_name'
)

In [50]:
# metab_betas

## Qatari cohort and QBB validation

In [51]:
yousri_2021 = pd.read_excel("../data/Yousri_2021.xlsx", sheet_name = 7, skiprows=[0], header=[0, 1, 2])

In [52]:
# print(yousri_2021.columns)

In [53]:
yousri_t2d = yousri_2021[[
    ('Metab', 'Metab', 'Metab'),
    ('T2D', 'Discovery', 'Beta'),
    ('T2D', 'Unnamed: 7_level_1', 'Replicated')
]].copy()

In [54]:
yousri_t2d.columns = yousri_t2d.columns.get_level_values(-1)

In [55]:
yousri_t2d.Replicated.value_counts()

Replicated
Yes          154
Not found     75
Name: count, dtype: int64

In [56]:
yousri_t2d_sig = yousri_t2d[yousri_t2d.Metab.isin(plot_labs)].copy()
yousri_t2d_sig.rename(columns={'Metab':'plot_name','Beta':'Qatari_T2D'}, inplace=True)

In [57]:
metab_betas = metab_betas.merge(
    yousri_t2d_sig[['plot_name', 'Qatari_T2D']], 
    on='plot_name', 
    how='left')

## Add determinants

In [58]:
diener_interactions = pd.read_excel("../data/diener_2022_gene_microbe_interactions.xlsx", sheet_name=2, header=0)

In [59]:
diener_interactions_label = diener_interactions[['BIOCHEMICAL_NAME','n_taxa','micro_r2', 'geno_r2', 'total', 'group']].copy()
diener_interactions_label.rename(columns={'BIOCHEMICAL_NAME':'plot_name'}, inplace=True)

In [60]:
metab_betas = metab_betas.merge(
    diener_interactions_label, 
    on='plot_name',
    how='left')

In [61]:
metab_betas.fillna({"SUPER_PATHWAY": "Unknown"}, inplace=True)
metab_betas.fillna({"SUB_PATHWAY": "Unknown"}, inplace=True)
metab_betas.fillna({"group": "N/A"}, inplace=True)

In [62]:
metab_betas.group = metab_betas.group.str.replace("hybrid","Hybrid")
metab_betas.group = metab_betas.group.str.replace("microbiome","Microbiome")
metab_betas.group = metab_betas.group.str.replace("genetics","Genetics")

In [63]:
metab_betas.rename(columns={'group':'Determinant_Association'}, inplace=True)

In [68]:
study_cols = ['abx_three', 'blood_sugar_meds',
              'metacardis_metabolic_syndrome', 'metacardis_severe_ob','metacardis_t2dm',
              'CGI_T2D_OR', 'IGT_OR', 'IFG_OR', 
              'EPIC_T2DM','EPIC_two_dx_OR',
              'li_t2dm',
              'Qatari_T2D']

In [72]:
metab_betas[study_cols].describe()

,abx_three,blood_sugar_meds,metacardis_metabolic_syndrome,metacardis_severe_ob,metacardis_t2dm,CGI_T2D_OR,IGT_OR,IFG_OR,Qatari_T2D
count,32.000000,13.000000,8.000000,4.000000,4.000000,18.000000,16.000000,4.000000,8.000000
mean,-0.170349,0.386852,-0.116356,-0.355400,-0.251317,1.284783,1.185619,1.015725,0.516150
std,0.207688,0.298985,0.386797,0.666650,0.614769,0.525265,0.371177,0.358853,0.136871
min,-0.359513,-0.395395,-0.523399,-0.829228,-0.694456,0.500500,0.662600,0.697300,0.298664
25%,-0.262694,0.319675,-0.393704,-0.714467,-0.556269,0.908300,0.764375,0.709000,0.435695
50%,-0.239928,0.337834,-0.282124,-0.610853,-0.484541,1.334750,1.264050,1.014000,0.514812
75%,-0.218525,0.544634,0.296105,-0.251786,-0.179590,1.451375,1.409100,1.320725,0.606864
max,0.354909,0.887619,0.416840,0.629332,0.658268,2.192500,1.731700,1.337600,0.709220


In [65]:
metab_betas.to_csv('../data/metabolite_meta_analysis_18Aug2026.csv', index=False)